## Imports

In [158]:
%load_ext autoreload
%autoreload 2

import pickle
import pandas as pd
import numpy as np

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [159]:
temp = ['ResNet-50-pretrained', 'ResNet-50-pretrained-clip', 'ResNet-50-scratch', 'ViT-b32-pretrained', 'ViT-b32-pretrained-clip', 'ViT-b32-scratch', 'ConvNeXt-base-pretrained','ConvNeXt-base-pretrained-clip','ConvNeXt-base-scratch']

In [160]:
for i in temp:
    print(f"'{i}': 0,")

'ResNet-50-pretrained': 0,
'ResNet-50-pretrained-clip': 0,
'ResNet-50-scratch': 0,
'ViT-b32-pretrained': 0,
'ViT-b32-pretrained-clip': 0,
'ViT-b32-scratch': 0,
'ConvNeXt-base-pretrained': 0,
'ConvNeXt-base-pretrained-clip': 0,
'ConvNeXt-base-scratch': 0,


In [161]:
!ls experiment_results*

experiment_results.csv
experiment_results_07212025_convnext_clip.csv
experiment_results_07212025_convnext_imagenet.csv
experiment_results_07212025_convnext_scratch.csv
experiment_results_07212025_resnet_clip.csv
experiment_results_07212025_resnet_imagenet.csv
experiment_results_07212025_resnet_scratch.csv
experiment_results_07212025_vit_clip.csv
experiment_results_07212025_vit_imagenet.csv
experiment_results_07212025_vit_scratch.csv
experiment_results_20250721.csv
experiment_results_convnext_clip.csv
experiment_results_convnext_imagenet.csv
experiment_results_convnext_scratch.csv
experiment_results_final_benchmark_07222025.csv
experiment_results_prelim_benchmark_07222025.csv
experiment_results_resnet_clip.csv
experiment_results_resnet_imagenet.csv
experiment_results_resnet_scratch.csv
experiment_results_vit_clip.csv
experiment_results_vit_imagenet.csv
experiment_results_vit_scratch.csv


In [162]:
files = [
    'experiment_results_07212025_convnext_scratch.csv',
    'experiment_results_07212025_convnext_clip.csv',
    'experiment_results_07212025_convnext_imagenet.csv',
    'experiment_results_07212025_resnet_scratch.csv',
    'experiment_results_07212025_resnet_clip.csv',
    'experiment_results_07212025_resnet_imagenet.csv',
    'experiment_results_07212025_vit_scratch.csv',
    'experiment_results_07212025_vit_clip.csv',
    'experiment_results_07212025_vit_imagenet.csv',
]
experiment_results = pd.concat((pd.read_csv(f) for f in files), ignore_index=True).drop_duplicates().reset_index(drop = True) # some jobs might have overlapped

In [163]:
experiment_results.shape

(811, 25)

In [165]:
# experiment_results.to_csv("experiment_results_final_benchmark_07222025.csv")

In [166]:
experiment_results['Pretraining Task'] = experiment_results['model'].apply(lambda x: 'ImageNet' if x[-1] == 'd' else ('None' if x[-1] == 'h' else 'CLIP'))
experiment_results['Model Architecture'] = experiment_results['model'].apply(lambda x: "-".join(x.split("-")[:2]))
experiment_results['early_stopping_point'] = experiment_results['train_loss_history'].apply(lambda x: sum([0 if i is None else 1 for i in eval(x)]))

In [167]:
# [0 if i is None else 1 for i in eval(experiment_results['train_loss_history'].iloc[0])]

In [168]:
experiment_results['early_stopping_point'].describe(percentiles = [0.95])

count    811.000000
mean      11.097411
std        4.566289
min        6.000000
50%       10.000000
95%       20.000000
max       47.000000
Name: early_stopping_point, dtype: float64

In [169]:
experiment_results.groupby('model').size()

model
ConvNeXt-base-pretrained         90
ConvNeXt-base-pretrained-clip    90
ConvNeXt-base-scratch            90
ResNet-50-pretrained             90
ResNet-50-pretrained-clip        90
ResNet-50-scratch                90
ViT-b32-pretrained               90
ViT-b32-pretrained-clip          90
ViT-b32-scratch                  91
dtype: int64

In [170]:
experiment_results.sort_values(by=['val_acc','val_roc_auc'], ascending = False).groupby(['Model Architecture','Pretraining Task']).first()[['experiment_id', 'val_acc', 'val_roc_auc', 'val_pr_auc', 'val_loss']]

experiment_id   val_acc  val_roc_auc  \
Model Architecture Pretraining Task                                         
ConvNeXt-base      CLIP                          2  0.846154     0.902327   
                   ImageNet                     46  0.803419     0.861604   
                   None                         56  0.658120     0.607165   
ResNet-50          CLIP                         51  0.811966     0.861911   
                   ImageNet                     64  0.811966     0.871555   
                   None                         17  0.794872     0.812921   
ViT-b32            CLIP                         50  0.837607     0.876607   
                   ImageNet                     38  0.820513     0.862829   
                   None                         62  0.692308     0.639927   

                                     val_pr_auc  val_loss  
Model Architecture Pretraining Task                        
ConvNeXt-base      CLIP                0.941156  0.490511  
                   ImageNet            0.919166  0.854010  
                   None                0.672875  0.664588  
ResNet-50          CLIP                0.910722  0.738837  
                   ImageNet            0.928704  0.536489  
                   None                0.852441  0.525099  
ViT-b32            CLIP                0.933870  0.609752  
                   ImageNet            0.922369  0.595384  
                   None                0.689035  0.640606

In [171]:
experiment_results.sort_values(by=['val_roc_auc','val_acc'], ascending = False).groupby(['Model Architecture','Pretraining Task']).first()[['experiment_id', 'val_roc_auc', 'val_acc', 'val_pr_auc', 'val_loss']]

experiment_id  val_roc_auc   val_acc  \
Model Architecture Pretraining Task                                         
ConvNeXt-base      CLIP                          2     0.902327  0.846154   
                   ImageNet                     76     0.879363  0.786325   
                   None                         23     0.615432  0.598291   
ResNet-50          CLIP                         51     0.861911  0.811966   
                   ImageNet                     46     0.884262  0.803419   
                   None                         17     0.812921  0.794872   
ViT-b32            CLIP                         38     0.893448  0.803419   
                   ImageNet                     38     0.862829  0.820513   
                   None                          2     0.663197  0.658120   

                                     val_pr_auc  val_loss  
Model Architecture Pretraining Task                        
ConvNeXt-base      CLIP                0.941156  0.490511  
                   ImageNet            0.930367  0.650654  
                   None                0.730050  0.671716  
ResNet-50          CLIP                0.910722  0.738837  
                   ImageNet            0.934866  0.640100  
                   None                0.852441  0.525099  
ViT-b32            CLIP                0.941792  0.525401  
                   ImageNet            0.922369  0.595384  
                   None                0.729974  0.643880